In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2, chi2_contingency, fisher_exact, norm
from scipy.stats import chisquare, poisson

## 📌 Question 1 : Chi-Square Goodness-of-Fit (Fair Die)

A die is rolled 120 times. Observed counts per face:

| Face | 1 | 2 | 3 | 4 | 5 | 6 |
|------|---|---|---|---|---|---|
| Observed | 18 | 22 | 17 | 25 | 20 | 18 |

Test at α = 0.05 whether the die is fair using chi-square goodness-of-fit. Compute χ², df, and the critical value.

*If the die is fair every face has equal probability 1/6, so under H₀ each face should appear n/6 = 20 times. The chi-square statistic measures how far the observed counts stray from those equal expected counts. A large χ² means the die is likely biased; a small one means the deviations are just chance. Since we are comparing one set of observed frequencies against a fully specified theoretical distribution (no parameters estimated from the data), df = k - 1 = 5.*

#### Step 1 : Hypotheses

$$H_0: p_1 = p_2 = p_3 = p_4 = p_5 = p_6 = \frac{1}{6} \quad \text{(die is fair)}$$
$$H_1: \text{at least one } p_i \neq \frac{1}{6} \quad \text{(die is not fair)}$$

#### Step 2 : Why This Test

We have one categorical variable (die face) with $k = 6$ categories and a fixed theoretical distribution under $H_0$. The chi-square goodness-of-fit test is designed exactly for this: it compares a single sample of observed frequencies against expected frequencies derived from the null distribution. All expected counts = 20 ≥ 5, so the large-sample approximation is valid.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}, \quad df = k - 1$$

where $O_i$ are observed counts and $E_i = n/k$ are expected counts under a fair die.

In [2]:
observed = np.array([18, 22, 17, 25, 20, 18])  # observed face counts
n = observed.sum()  # total rolls = 120
alpha = 0.05
k = len(observed)  # number of categories

In [3]:
expected = np.array([n/k] * k)  # expected counts under fair die
chi_stat = sum((observed - expected)**2 / expected)

print('Expected counts :', expected)
print('chi_stat        :', round(chi_stat, 4))

Expected counts : [20. 20. 20. 20. 20. 20.]
chi_stat        : 2.3


#### Step 4 : Degrees of Freedom

$$df = k - 1 = 6 - 1 = 5$$

No parameters are estimated from the data (the fair-die probabilities are fully specified), so we lose only 1 degree of freedom for the total constraint $\sum O_i = n$.

In [4]:
df = k - 1  # degrees of freedom = 5
print('df :', df)

df : 5


#### Step 5 : Critical Value Method

In [5]:
chi_crit = chi2.ppf(1-alpha, df)  # right-tail critical value
print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))

chi_stat : 2.3
chi_crit : 11.0705


**chi_stat (2.30) < chi_crit (11.071) → Fail to Reject H₀**

> Note: chi-square goodness-of-fit is always a **right-tailed** test - only an unusually *large* χ² is evidence against H₀. The critical value uses `chi2.ppf(1 - alpha, df)`, not `1 - alpha/2`.

#### Step 6 : p-value Method

In [6]:
pval = chi2.sf(chi_stat, df)  # right-tail p-value
print('p-val :', round(pval, 4))

p-val : 0.8063


**p-val (0.806) > 0.05 → Fail to Reject H₀**

#### Final Conclusion

| Metric | Value |
|---|---|
| Total rolls $n$ | 120 |
| Categories $k$ | 6 |
| Expected per face | 20 |
| $\chi^2$ statistic | 2.30 |
| Degrees of freedom | 5 |
| $\chi^2$ critical (right-tail, α = 0.05) | 11.071 |
| p-value | 0.806 |
| Decision | **Fail to Reject H₀** |

There is insufficient evidence to conclude the die is unfair. The observed counts (18, 22, 17, 25, 20, 18) are consistent with random variation expected from a fair die (χ² = 2.30, p = 0.806).

## 📌 Question 2 : Chi-Square Test of Independence (Gender × Coffee Preference)

A survey of 200 people records gender and coffee preference (Yes / No):

|  | Coffee: Yes | Coffee: No | Row Total |
|---|---|---|---|
| Male | 60 | 40 | 100 |
| Female | 55 | 45 | 100 |
| **Col Total** | **115** | **85** | **200** |

Test independence at α = 0.05. Compute the expected frequency for each cell and verify that all expected counts are ≥ 5.

*If gender and coffee preference are independent, the proportion of coffee drinkers should be the same for males and females. Under H₀ we estimate each cell's expected count from the marginal totals alone: E = (row total × col total) / n. A large χ² signals that the observed cell counts deviate more than chance would produce if the two variables were truly unrelated.*

#### Step 1 : Hypotheses

$$H_0: \text{Gender and coffee preference are independent}$$
$$H_1: \text{Gender and coffee preference are dependent (associated)}$$

#### Step 2 : Why This Test

We have two categorical variables (gender: 2 levels; coffee preference: 2 levels) measured on a single sample of $n = 200$ individuals. The chi-square test of independence is the appropriate test: it checks whether the joint distribution of the two variables equals the product of their marginal distributions. With a 2 × 2 contingency table, $df = (r-1)(c-1) = 1$. All expected counts must be ≥ 5 for the large-sample approximation to hold; we verify this in Step 3.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad E_{ij} = \frac{(\text{row}_i \text{ total}) \times (\text{col}_j \text{ total})}{n}$$

In [7]:
alpha = 0.05

O = np.array([[60, 40],   # [Male-Yes,   Male-No]
              [55, 45]])  # [Female-Yes, Female-No]

In [8]:
row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

# E_ij = (row_i * col_j) / n
E = np.outer(row_total, col_total) / n

print('Row totals :', row_total)
print('Col totals :', col_total)
print('n          :', n)
print()
print('Expected counts E:')
print(E)
print()
print('All expected counts >= 5:', (E >= 5).all())

Row totals : [100 100]
Col totals : [115  85]
n          : 200

Expected counts E:
[[57.5 42.5]
 [57.5 42.5]]

All expected counts >= 5: True


In [9]:
chi_stat = ((O - E)**2 / E).sum()
print('chi_stat :', round(chi_stat, 4))

chi_stat : 0.5115


#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (2 - 1)(2 - 1) = 1$$

For an $r \times c$ contingency table we lose one degree of freedom per row and per column after fixing the marginal totals, leaving $(r-1)(c-1)$ free cells.

In [10]:
r, c = O.shape
df = (r-1) * (c-1)  # degrees of freedom = 1
print('df :', df)

df : 1


#### Step 5 : Critical Value Method

In [11]:
chi_crit = chi2.ppf(1-alpha, df)  # right-tail critical value
print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))

chi_stat : 0.5115
chi_crit : 3.8415


**chi_stat (0.511) < chi_crit (3.841) -- Fail to Reject H₀**

> Note: the chi-square test of independence is always **right-tailed**. Only an unusually large χ² constitutes evidence of association. The critical value uses `chi2.ppf(1 - alpha, df)`.

#### Step 6 : p-value Method

In [12]:
pval = chi2.sf(chi_stat, df)  # right-tail p-value
print('p-val :', round(pval, 4))

p-val : 0.4745


**p-val (0.474) > 0.05 -- Fail to Reject H₀**

#### Final Conclusion

**Expected counts (all ≥ 5 -- large-sample approximation is valid):**

|  | Coffee: Yes | Coffee: No |
|---|---|---|
| Male | 57.5 | 42.5 |
| Female | 57.5 | 42.5 |

**Summary:**

| Metric | Value |
|---|---|
| Sample size $n$ | 200 |
| Table dimensions | 2 × 2 |
| $\chi^2$ statistic | 0.511 |
| Degrees of freedom | 1 |
| $\chi^2$ critical (right-tail, α = 0.05) | 3.841 |
| p-value | 0.474 |
| Decision | **Fail to Reject H₀** |

There is insufficient evidence to conclude that gender and coffee preference are associated. The small χ² (0.511) and large p-value (0.474) indicate that the observed cell counts are well within the range of random variation expected if the two variables were independent (χ² = 0.511, p = 0.474).

## 📌 Question 3 : Fisher's Exact Test vs Chi-Square (Clinical Trial)

A clinical trial with small cell counts records outcomes for a new drug vs placebo:

|  | Cured | Not Cured | Row Total |
|---|---|---|---|
| New Drug | 8 | 2 | 10 |
| Placebo | 4 | 6 | 10 |
| **Col Total** | **12** | **8** | **20** |

**(a)** Run chi-square without and with Yates' continuity correction.  
**(b)** Run Fisher's exact test.  
**(c)** Compare all three results.  
**(d)** State the rule: when must you use Fisher's exact test?

*The key issue here is small expected cell counts. When any expected frequency falls below 5, the chi-square large-sample approximation breaks down and Fisher's exact test -- which enumerates all possible tables with the same marginals -- is the correct choice. Yates' continuity correction partially compensates for the discreteness of the data but is still an approximation.*

#### Step 1 : Hypotheses

$$H_0: \text{Treatment and outcome are independent (drug has no effect)}$$
$$H_1: \text{Treatment and outcome are dependent (drug has an effect)}$$

#### Step 2 : Why This Test Matters Here

We have a 2x2 contingency table with only n = 20 observations. Before choosing a test we must check the expected cell counts under H0. If any expected count is below 5, the chi-square approximation is unreliable and Fisher's exact test is required.

#### Step 3 : Observed Table and Expected Counts

$$E_{ij} = \frac{(\text{row}_i \text{ total}) \times (\text{col}_j \text{ total})}{n}$$

In [13]:
alpha = 0.05

O = np.array([[8, 2],
              [4, 6]])

row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

E = np.outer(row_total, col_total) / n

print('Row totals :', row_total)
print('Col totals :', col_total)
print('n          :', n)
print()
print('Expected counts E:')
print(E)
print()
print('Any expected count < 5 :', (E < 5).any())

Row totals : [10 10]
Col totals : [12  8]
n          : 20

Expected counts E:
[[6. 4.]
 [6. 4.]]

Any expected count < 5 : True


**Expected counts:**

|  | Cured | Not Cured |
|---|---|---|
| New Drug | 6.0 | 4.0 |
| Placebo | 6.0 | 4.0 |

**Two cells have expected count = 4.0 < 5.** The chi-square large-sample approximation is not strictly valid here. We proceed with both chi-square variants for comparison, but Fisher's exact test is the correct choice.

#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (2 - 1)(2 - 1) = 1$$

In [14]:
df = (O.shape[0]-1) * (O.shape[1]-1)
print('df :', df)

df : 1


#### Step 5 (a) : Chi-Square Without Yates' Correction

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [15]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1-alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat (no correction) :', round(chi_stat, 4))
print('chi_crit                 :', round(chi_crit, 4))
print('p-value                  :', round(pval, 4))

chi_stat (no correction) : 3.3333
chi_crit                 : 3.8415
p-value                  : 0.0679


**chi_stat (3.3333) < chi_crit (3.8415) -- Fail to Reject H0**  
**p-value (0.0679) > 0.05 -- Fail to Reject H0**

Without correction, the association is not significant at the 5% level, but it is borderline (p = 0.068).

#### Step 5 (b) : Chi-Square With Yates' Continuity Correction

Yates' correction subtracts 0.5 from each |O - E| before squaring, reducing the test statistic to partially compensate for the discreteness of count data:

$$\chi^2_{\text{Yates}} = \sum_{i,j} \frac{(|O_{ij} - E_{ij}| - 0.5)^2}{E_{ij}}$$

In [16]:
chi_stat_yates = ((np.abs(O - E) - 0.5)**2 / E).sum()
pval_yates = chi2.sf(chi_stat_yates, df)

print('chi_stat (Yates) :', round(chi_stat_yates, 4))
print('chi_crit         :', round(chi_crit, 4))
print('p-value (Yates)  :', round(pval_yates, 4))

chi_stat (Yates) : 1.875
chi_crit         : 3.8415
p-value (Yates)  : 0.1709


**chi_stat_yates (1.8750) < chi_crit (3.8415) -- Fail to Reject H0**  
**p-value (0.1709) > 0.05 -- Fail to Reject H0**

With Yates' correction the statistic drops considerably and the p-value rises to 0.171, making the result even less significant.

#### Step 5 (c) : Fisher's Exact Test

Fisher's exact test computes the exact probability of observing a table as extreme as (or more extreme than) the one observed, given fixed marginal totals. It makes no large-sample approximation and is valid for any sample size.

The p-value is the sum of hypergeometric probabilities for all tables at least as extreme as the observed one.

In [17]:
oddsratio, fisher_p = fisher_exact(O, alternative='two-sided')

print('Odds Ratio :', round(oddsratio, 4))
print('p-value    :', round(fisher_p, 4))

Odds Ratio : 6.0
p-value    : 0.1698


**p-value (0.1698) > 0.05 -- Fail to Reject H0**

Fisher's exact test gives p = 0.170. There is insufficient evidence to conclude the drug is effective at the 5% significance level.

The odds ratio of 6.0 means patients on the new drug were 6 times more likely to be cured than those on placebo, but with only 20 observations this effect is not statistically significant.

#### Step 6 : Comparison of All Three Results

| Method | Test Statistic | p-value | Decision |
|---|---|---|---|
| Chi-square (no correction) | 3.3333 | 0.0679 | Fail to Reject H0 |
| Chi-square (Yates correction) | 1.8750 | 0.1709 | Fail to Reject H0 |
| Fisher's Exact Test | -- | 0.1698 | Fail to Reject H0 |

All three methods lead to the same decision, but the p-values differ substantially. The uncorrected chi-square (p = 0.068) is the most liberal and comes closest to the 5% threshold. Yates' correction and Fisher's exact test agree closely (p ~ 0.170), which is expected: Fisher's exact test is the gold standard for small samples and Yates' correction is designed to approximate it.

> The uncorrected chi-square overstates significance here because the large-sample approximation is unreliable when expected counts fall below 5.

#### Step 7 (d) : When Must You Use Fisher's Exact Test?

Use Fisher's exact test when **any** expected cell count in the contingency table is less than 5.

More precisely:

| Condition | Recommended Test |
|---|---|
| All expected counts >= 10 | Chi-square (no correction) is reliable |
| Any expected count between 5 and 10 | Chi-square with Yates' correction |
| Any expected count < 5 | Fisher's exact test (required) |
| Total sample size n < 20 | Fisher's exact test (always preferred) |

In this problem, two cells have expected count = 4.0 < 5, so Fisher's exact test is the appropriate choice. The chi-square results are shown for comparison only.

#### Final Conclusion

| Metric | Value |
|---|---|
| Sample size n | 20 |
| Table dimensions | 2 x 2 |
| Minimum expected count | 4.0 (below threshold of 5) |
| Chi-square statistic (no correction) | 3.3333 |
| Chi-square p-value (no correction) | 0.0679 |
| Chi-square statistic (Yates) | 1.8750 |
| Chi-square p-value (Yates) | 0.1709 |
| Fisher's exact p-value | 0.1698 |
| Odds ratio | 6.0000 |
| Decision | **Fail to Reject H0** |

Because two expected cell counts (4.0) fall below 5, Fisher's exact test is the correct test. All three methods agree: there is insufficient evidence at alpha = 0.05 to conclude that the drug is more effective than placebo (Fisher p = 0.170). The odds ratio of 6.0 suggests a clinically meaningful trend, but the small sample size (n = 20) leaves the result statistically inconclusive.

## 📌 Question 4 : Chi-Square + Standardized Residuals (Age × Campaign)

A marketing survey of 413 people. We want to know: does campaign preference depend on age group?

|  | Campaign A | Campaign B | Campaign C | Total |
|---|---|---|---|---|
| Age 18–30 | 45 | 38 | 52 | 135 |
| Age 31–50 | 62 | 70 | 58 | 190 |
| Age 51+   | 28 | 35 | 25 | 88 |
| **Total** | **135** | **143** | **135** | **413** |

**(a)** Chi-square test of independence  
**(b)** Standardized residuals for each cell  
**(c)** Which cells are driving the result?  
**(d)** What |SR| threshold do we use at α = 0.05?

*The chi-square test will tell us whether there is an overall association between age and preference. But it gives us just one number - it cannot tell us which specific age-campaign combination is unusual. That is what standardized residuals are for. Each SR value is basically a z-score for one cell: how far is the observed count from what we expected, in standard deviation units. A cell with |SR| > 2 is the one worth paying attention to.*

#### Step 1 : Hypotheses

$$H_0: \text{Age group and campaign preference are independent}$$
$$H_1: \text{They are associated}$$

#### Step 2 : Data Setup

In [18]:
alpha = 0.05

O = np.array([
    [45, 38, 52],   # Age 18-30
    [62, 70, 58],   # Age 31-50
    [28, 35, 25]    # Age 51+
])

row_totals = O.sum(axis=1)
col_totals = O.sum(axis=0)
n = O.sum()

print('Row totals :', row_totals)
print('Col totals :', col_totals)
print('n          :', n)

Row totals : [135 190  88]
Col totals : [135 143 135]
n          : 413


#### Step 3 : Expected Counts

$$E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{n}$$

*If age and preference are independent, the expected count is just row proportion × column proportion × n. np.outer multiplies every row total with every column total in one go - same as doing it manually for each cell.*

In [19]:
E = np.outer(row_totals, col_totals) / n

print('Expected counts:')
print(E.round(2))
print()
print('All counts >= 5 :', (E >= 5).all())

Expected counts:
[[44.13 46.74 44.13]
 [62.11 65.79 62.11]
 [28.77 30.47 28.77]]

All counts >= 5 : True


#### Step 4 : Degrees of Freedom

$$df = (r-1)(c-1) = (3-1)(3-1) = 4$$

In [20]:
rows, cols = O.shape
df = (rows-1) * (cols-1)
print('df :', df)

df : 4


#### Step 5 (a) : Chi-Square Test

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

In [21]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1-alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))
print('p-value  :', round(pval, 4))

chi_stat : 4.7851
chi_crit : 9.4877
p-value  : 0.3101


**chi_stat (4.7851) < chi_crit (9.4877) → Fail to Reject H₀**  
**p-value (0.3101) > 0.05 → Fail to Reject H₀**

#### Step 6 (b) : Standardized Residuals

$$SR_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

*Think of each SR as a z-score for that one cell. Positive means the observed count is higher than expected, negative means lower. The sqrt(E) in the denominator just scales things so all cells are comparable regardless of size.*

In [22]:
SR = (O - E) / np.sqrt(E)
print(SR.round(2))

[[ 0.13 -1.28  1.18]
 [-0.01  0.52 -0.52]
 [-0.14  0.82 -0.7 ]]


#### Step 7 (c) : Which Cells Stand Out?

In [23]:
# check each cell manually
age_labels  = ["Age 18-30", "Age 31-50", "Age 51+"]
camp_labels = ["Campaign A", "Campaign B", "Campaign C"]

print("Cells with |SR| > 2 :")
found = False
for i in range(3):
    for j in range(3):
        if abs(SR[i][j]) > 2:
            print(f"  {age_labels[i]} x {camp_labels[j]} : SR = {SR[i][j]:.4f}")
            found = True
if not found:
    print("  None - no cell is individually unusual")

Cells with |SR| > 2 :
  None - no cell is individually unusual


**No cells have |SR| > 2.** The largest is 1.28 (Age 18–30 × Campaign B), which is well within normal range. This makes sense - the overall test was also non-significant.

#### Step 8 (d) : Why |SR| > 2?

*SR values follow a standard normal distribution under H₀. For a 5% significance level, the cutoff is ±1.96 ≈ 2. So |SR| > 2 just means that cell is significant at the ~5% level on its own. For bigger tables (lots of cells), some people use |SR| > 3 to be stricter.*

| |SR| | What it means |
|---|---|
| < 1 | very close to expected, nothing unusual |
| 1 – 2 | mild deviation, not flagged |
| > 2 | notable at α = 0.05 |
| > 3 | strong, used as stricter threshold |

#### Final Conclusion

| Metric | Value |
|---|---|
| n | 413 |
| Table | 3 × 3 |
| All expected counts ≥ 5 | ✓ |
| χ² statistic | 4.7851 |
| df | 4 |
| χ² critical (α = 0.05) | 9.4877 |
| p-value | 0.3101 |
| Decision | **Fail to Reject H₀** |
| Cells with |SR| > 2 | None |

Age group and campaign preference are independent. No individual cell shows a meaningful deviation from what we expected under independence (χ² = 4.785, p = 0.310).

## 📌 Question 5 : Chi-Square Goodness-of-Fit (Snack Flavor Preference)

A snack company claims customer preference for 4 flavors is equally distributed.

| Flavor | Classic | Cheese | Spicy | BBQ |
|--------|---------|--------|-------|-----|
| Observed Sales | 52 | 47 | 61 | 40 |

Perform a chi-square goodness-of-fit test at α = 0.05.

**(a)** Perform the full chi-square test manually.

**(b)** Solve again using `scipy.stats.chisquare()`.

**(c)** Find which flavor(s) contributed most to the chi-square statistic.

*The company's claim translates directly into a null hypothesis: each of the 4 flavors should account for exactly 1/4 of all sales. Under H₀ the expected count for every flavor is n/4. A large χ² means the observed sales deviate more than chance alone would explain. Since we are comparing one observed distribution against a fully specified theoretical one (no parameters estimated from data), df = k - 1 = 3.*

#### Step 1 : Hypotheses

$$H_0: p_{\text{Classic}} = p_{\text{Cheese}} = p_{\text{Spicy}} = p_{\text{BBQ}} = \frac{1}{4} \quad \text{(preference is equally distributed)}$$
$$H_1: \text{at least one } p_i \neq \frac{1}{4} \quad \text{(preference is not equally distributed)}$$

#### Step 2 : Why This Test

We have one categorical variable (flavor) with $k = 4$ categories and a fully specified theoretical distribution under $H_0$ (equal probabilities). The chi-square goodness-of-fit test is designed exactly for this: it compares a single sample of observed frequencies against expected frequencies derived from the null distribution. Expected count per flavor = $200/4 = 50 \geq 5$, so the large-sample approximation is valid.

#### Step 3 : Test Statistic

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}, \quad df = k - 1$$

where $O_i$ are observed sales and $E_i = n/k$ are expected counts under equal preference.

In [24]:
O = np.array([52,47,61,40])

In [25]:
alpha =  0.05

In [26]:
n = O.sum()
n

np.int64(200)

In [27]:
E = np.array([n/4] * 4)
E

array([50., 50., 50., 50.])

In [28]:
chi_stat = sum((O-E)**2/E)
chi_stat

np.float64(4.68)

#### Step 4 : Degrees of Freedom

$$df = k - 1 = 4 - 1 = 3$$

No parameters are estimated from the data (equal probabilities are fully specified under $H_0$), so we lose only 1 degree of freedom for the constraint $\sum O_i = n$.

In [29]:
df = 3

#### Step 5 : Critical Value Method

In [30]:
chi_crit= chi2.ppf(1-alpha,df)
chi_crit

np.float64(7.814727903251179)

**chi_stat (4.68) < chi_crit (7.815) - Fail to Reject H₀**

> Note: chi-square goodness-of-fit is always a **right-tailed** test - only an unusually *large* χ² is evidence against H₀. The critical value uses `chi2.ppf(1 - alpha, df)`, not `1 - alpha/2`.

#### Step 6 : p-value Method

In [31]:
pval= chi2.sf(chi_stat,df)
pval

np.float64(0.19678574982211994)

**p-val (0.197) > 0.05 - Fail to Reject H₀**

#### Step 7 (b) : Verify with `scipy.stats.chisquare()`

`chisquare()` assumes equal expected frequencies by default when no `f_exp` is passed, so it matches our manual calculation exactly.

In [32]:
#part b
chi_stat, pval = chisquare(O)
chi_stat, pval

(np.float64(4.68), np.float64(0.19678574982211994))

#### Step 8 (c) : Which Flavor Contributed Most?

*Each term $(O_i - E_i)^2 / E_i$ is that flavor's individual contribution to χ². The flavor with the largest term is the one pulling the statistic the furthest from zero.*

In [33]:
contributors = (O-E)**2/E
contributors

array([0.08, 0.18, 2.42, 2.  ])

In [34]:
largest = np.argmax(contributors)
largest

np.int64(2)

In [35]:
categories = ["Classic", "Cheese", "Spicy", "BBQ"]
print(categories[largest])

Spicy


#### Final Conclusion

| Metric | Value |
|---|---|
| Total sales $n$ | 200 |
| Categories $k$ | 4 |
| Expected per flavor | 50 |
| $\chi^2$ statistic | 4.68 |
| Degrees of freedom | 3 |
| $\chi^2$ critical (right-tail, α = 0.05) | 7.815 |
| p-value | 0.197 |
| Decision | **Fail to Reject H₀** |
| Largest contributor | **Spicy** |

There is insufficient evidence to conclude that customer preference is unequally distributed across the four flavors (χ² = 4.68, p = 0.197). Although Spicy had the highest observed sales (61) and contributed the most to χ², the deviation is within the range of random variation expected under equal preference.

## 📌 Question 6 : Chi-Square Test of Homogeneity + Post-Hoc Analysis (Teaching Methods)

Three different teaching methods are tested on separate groups of students. After the course, students are classified by performance level.

|  | High | Medium | Low | Row Total |
|---|---|---|---|---|
| Method A | 48 | 32 | 20 | 100 |
| Method B | 30 | 50 | 20 | 100 |
| Method C | 22 | 38 | 40 | 100 |
| **Col Total** | **100** | **120** | **80** | **300** |

**(a)** Perform a chi-square test of homogeneity manually at α = 0.05.

**(b)** Compute all expected frequencies.

**(c)** Verify whether chi-square assumptions are satisfied.

**(d)** Compute standardized residuals for every cell.

**(e)** Identify which teaching method-performance combinations drive the significance.

**(f)** Compute Cramér's V and interpret the strength of association.

*The chi-square test of homogeneity asks whether the distribution of a categorical outcome (performance level) is the same across several independent groups (teaching methods). It uses the same χ² formula as the test of independence, but the sampling design is different: here, each row is a separately drawn sample of 100 students. If the methods are equally effective, performance should follow the same proportions in every group. A significant χ² tells us the distributions differ somewhere; standardized residuals then pinpoint which specific cells are responsible.*

#### Step 1 : Hypotheses

$$H_0: \text{The distribution of performance is the same across all three teaching methods}$$
$$H_1: \text{At least one teaching method has a different performance distribution}$$

#### Step 2 : Data Setup

In [36]:
alpha = 0.05

O = np.array([
    [48, 32, 20],   # Method A
    [30, 50, 20],   # Method B
    [22, 38, 40]])  # Method C

row_totals = O.sum(axis=1)
col_totals = O.sum(axis=0)
n = O.sum()

print('Row totals :', row_totals)
print('Col totals :', col_totals)
print('n          :', n)

Row totals : [100 100 100]
Col totals : [100 120  80]
n          : 300


#### Step 3 (b) : Expected Counts

$$E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{n}$$

*Under H₀ each cell's expected count is the product of its row and column marginals divided by n. `np.outer` computes all nine cells at once -- the same as doing it manually for each cell.*

In [37]:
E = np.outer(row_totals, col_totals) / n

print('Expected counts:')
print(E.round(2))

Expected counts:
[[33.33 40.   26.67]
 [33.33 40.   26.67]
 [33.33 40.   26.67]]


#### Step 3 (c) : Verify Chi-Square Assumptions

In [38]:
print('All expected counts >= 5 :', (E >= 5).all())
print('Minimum expected count   :', E.min().round(2))

All expected counts >= 5 : True
Minimum expected count   : 26.67


All nine expected counts are well above 5, so the large-sample chi-square approximation is valid.

#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (3 - 1)(3 - 1) = 4$$

With 3 rows and 3 columns, fixing the marginal totals leaves $(r-1)(c-1) = 4$ free cells.

In [39]:
rows, cols = O.shape
df = (rows - 1) * (cols - 1)
print('df :', df)

df : 4


#### Step 5 (a) : Chi-Square Test

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [40]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1 - alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))
print('p-value  :', round(pval, 4))

chi_stat : 24.84
chi_crit : 9.4877
p-value  : 0.0001


**chi_stat (24.84) > chi_crit (9.4877) → Reject H₀**  
**p-value (0.0001) < 0.05 → Reject H₀**

The overall test is significant: the distribution of performance levels is not the same across all three teaching methods.

#### Step 6 (d) : Standardized Residuals

$$SR_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

*Each SR is a z-score for that one cell. Positive means the observed count is higher than expected; negative means lower. Values with |SR| > 2 flag cells that individually deviate significantly at the ~5% level.*

In [41]:
SR = (O - E) / np.sqrt(E)
print('Standardized Residuals:')
print(SR.round(4))

Standardized Residuals:
[[ 2.5403 -1.2649 -1.291 ]
 [-0.5774  1.5811 -1.291 ]
 [-1.963  -0.3162  2.582 ]]


#### Step 7 (e) : Which Cells Drive the Significance?

In [42]:
method_labels = ['Method A', 'Method B', 'Method C']
perf_labels   = ['High', 'Medium', 'Low']

print('Cells with |SR| > 2 :')
found = False
for i in range(rows):
    for j in range(cols):
        if abs(SR[i, j]) > 2:
            print(f'  {method_labels[i]} x {perf_labels[j]} : SR = {SR[i, j]:.4f}')
            found = True
if not found:
    print('  None')

Cells with |SR| > 2 :
  Method A x High : SR = 2.5403
  Method C x Low : SR = 2.5820


Cells with |SR| > 2 indicate where the observed counts deviate most from what homogeneity would predict:

- **Method A × High** (SR = 2.54): Method A produces more high performers than expected.
- **Method C × Low** (SR = 2.58): Method C produces more low performers than expected.

Method A is associated with stronger performance; Method C is associated with weaker performance.

#### Step 8 (f) : Cramér's V

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

Cramér's V ranges from 0 (no association) to 1 (perfect association). For a 3 × 3 table, the conventional benchmarks are: V < 0.1 weak, 0.1-0.3 moderate, > 0.3 strong.

In [43]:
k = min(O.shape)  # min(r, c) = 3
V = np.sqrt(chi_stat / (n * (k - 1)))
print('Cramer\'s V :', round(V, 4))

Cramer's V : 0.2035


A Cramér's V of 0.2035 indicates a **moderate association** between teaching method and performance level.

#### Final Conclusion

| Metric | Value |
|---|---|
| Sample size $n$ | 300 |
| Table dimensions | 3 × 3 |
| All expected counts ≥ 5 | ✓ |
| $\chi^2$ statistic | 24.84 |
| Degrees of freedom | 4 |
| $\chi^2$ critical (α = 0.05) | 9.4877 |
| p-value | 0.0001 |
| Decision | **Reject H₀** |
| Cells with \|SR\| > 2 | Method A × High, Method C × Low |
| Cramér's V | 0.2035 (moderate) |

There is significant evidence that performance distributions differ across teaching methods (χ² = 24.84, p = 0.0001). Method A is associated with higher performance, while Method C is associated with weaker performance. Cramér's V = 0.2035 indicates a moderate practical effect.

## 📌 Question 7 : Cramér's V and Contingency Coefficient C

Given: $\chi^2 = 18.4$, $n = 250$, from a $3 \times 4$ contingency table.

**(a)** Compute Cramér's V.  
**(b)** Compute the contingency coefficient $C$.  
**(c)** Why is $C$ problematic for comparing across tables of different dimensions?  
**(d)** Interpret $V$: what are the small, medium, and large thresholds for a $3 \times 4$ table?

*Both $V$ and $C$ are effect-size measures derived from $\chi^2$, but they behave very differently. Cramér's $V$ is normalized by the minimum dimension of the table, so it always ranges from 0 to 1 regardless of table size. The contingency coefficient $C$ is not properly normalized: its upper bound $C_{\text{max}} < 1$ and depends on $\min(r, c)$, making it impossible to compare values across tables of different shapes.*

#### Step 1 : Given Values

$$\chi^2 = 18.4, \quad n = 250, \quad r = 3 \text{ rows}, \quad c = 4 \text{ columns}$$

In [44]:
chi_stat = 18.4
n = 250
row = 3
col = 4

#### Step 2 (a) : Cramér's V

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

*We divide by $\min(r, c) - 1$ so that $V$ reaches 1.0 when the association is perfect, regardless of table dimensions. For a $3 \times 4$ table, $\min(r, c) = 3$, so the denominator is $n \cdot 2$.*

In [45]:
k = min(row, col)          # min(r, c) = 3
V = np.sqrt(chi_stat / (n * (k - 1)))
print("Cramer's V :", round(V, 4))

Cramer's V : 0.1918


#### Step 3 (b) : Contingency Coefficient $C$

$$C = \sqrt{\frac{\chi^2}{\chi^2 + n}}$$

*$C$ is bounded above by $C_{\text{max}} = \sqrt{1 - 1/\min(r,c)}$, which is less than 1 for any finite table. We compute $C$ and its upper bound together so the limitation is immediately visible.*

In [46]:
C = np.sqrt(chi_stat / (chi_stat + n))
C_max = np.sqrt(1 - 1/k)

print(f"C       : {C:.4f}")
print(f"C_max   : {C_max:.4f}  (upper bound for a {row}x{col} table)")

C       : 0.2618
C_max   : 0.8165  (upper bound for a 3x4 table)


#### Step 4 (c) : Why $C$ Is Problematic for Cross-Table Comparisons

The maximum value $C$ can reach is not 1 but:

$$C_{\text{max}} = \sqrt{1 - \frac{1}{\min(r, c)}}$$

This ceiling depends on the table's smaller dimension:

| Table size | $\min(r,c)$ | $C_{\text{max}}$ |
|---|---|---|
| $2 \times 2$ | 2 | 0.707 |
| $3 \times 3$ | 3 | 0.816 |
| $3 \times 4$ | 3 | 0.816 |
| $5 \times 5$ | 5 | 0.894 |

A $C = 0.70$ from a $2 \times 2$ table represents near-perfect association (close to its ceiling of 0.707), while the same $C = 0.70$ from a $5 \times 5$ table represents only a moderate association (far from its ceiling of 0.894). Raw $C$ values cannot be compared across tables of different shapes. Cramér's $V$ does not have this problem because it is normalized to always range from 0 to 1.

#### Step 5 (d) : Interpreting Cramér's V for a $3 \times 4$ Table

Cohen's conventional benchmarks for $V$ depend on $df^* = \min(r, c) - 1$. For this table $df^* = 2$:

| Effect size | $V$ threshold ($df^* = 2$) |
|---|---|
| Small | 0.07 |
| Medium | 0.21 |
| Large | 0.35 |

*These thresholds come from Cohen (1988): small = $0.10/\sqrt{df^*}$, medium = $0.30/\sqrt{df^*}$, large = $0.50/\sqrt{df^*}$.*

In [47]:
print(f"V = {V:.3f}, C = {C:.3f}, C_max = {C_max:.3f}")

V = 0.192, C = 0.262, C_max = 0.816


#### Final Conclusion

| Metric | Value |
|---|---|
| $\chi^2$ statistic | 18.4 |
| Sample size $n$ | 250 |
| Table dimensions | $3 \times 4$ |
| $\min(r, c)$ | 3 |
| Cramér's $V$ | 0.1918 |
| Contingency coefficient $C$ | 0.2618 |
| $C_{\text{max}}$ (upper bound) | 0.8165 |
| Effect size (Cohen, $df^* = 2$) | **Small to medium** |

Cramér's $V \approx 0.19$ indicates a small-to-medium association between the two categorical variables. The contingency coefficient $C \approx 0.26$ cannot be interpreted on a fixed 0-to-1 scale because its ceiling for a $3 \times 4$ table is only 0.82; comparing it to $C$ values from tables of different dimensions would be misleading. Cramér's $V$ is the preferred effect-size measure for chi-square tests.

## 📌 Question 8 : Chi-Square Test of Independence + Residual Analysis

A multinational tech company studies whether preferred work mode depends on employee department.

|  | Remote | Hybrid | Office |
|---|---|---|---|
| Engineering | 84 | 51 | 25 |
| Marketing | 39 | 67 | 44 |
| Finance | 26 | 48 | 56 |
| HR | 18 | 29 | 43 |

**(a)** Perform the chi-square test manually.

**(b)** Solve again using `scipy.stats.chi2_contingency()`.

**(c)** Compute standardized residuals for every cell.

**(d)** Identify which cells contribute most to the chi-square statistic.

**(e)** Compute Cramér's V.

**(f)** Interpret whether the association is weak, moderate, or strong.

*The chi-square test tells us whether work mode preference is distributed the same way across all departments. A significant result just means the distributions differ somewhere; standardized residuals then show which department-work-mode combinations are responsible. Cramer's V gives the effect size on a 0-to-1 scale.*

#### Step 1 : Hypotheses

$$H_0: \text{Employee department and work mode preference are independent}$$
$$H_1: \text{Employee department and work mode preference are not independent (associated)}$$

#### Step 2 : Why This Test

We have two categorical variables (department: 4 levels, work mode: 3 levels) on a single sample of $n = 530$ employees. The chi-square test of independence checks whether the two variables are related or not. With a 4 x 3 table, $df = (r-1)(c-1) = 6$. All expected counts should be >= 5 for the large-sample approximation to hold; we check this after computing E.

#### Step 3 : Data Setup and Expected Counts

$$E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{n}$$

*If department and work mode are independent, each cell's expected count is determined by the marginal totals alone. np.outer handles all 12 cells at once.*

In [48]:
O = np.array([[84, 51, 25],
              [39, 67, 44],
              [26, 48, 56],
              [18, 29, 43]])

In [49]:
alpha = 0.05

In [50]:
row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

In [51]:
r, c = O.shape

In [52]:
E = np.outer(row_total, col_total) / n
E

array([[50.41509434, 58.86792453, 50.71698113],
       [47.26415094, 55.18867925, 47.54716981],
       [40.96226415, 47.83018868, 41.20754717],
       [28.35849057, 33.11320755, 28.52830189]])

All 12 expected counts are above 5, so the large-sample approximation is valid.

#### Step 4 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (4 - 1)(3 - 1) = 6$$

In [53]:
df = (r-1) * (c-1)
df

6

#### Step 5 (a) : Chi-Square Test (manual)

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [54]:
chi_stat = ((O - E)**2 / E).sum()
chi_stat

np.float64(63.11414381079814)

#### Step 6 : Critical Value and p-value

In [55]:
chi_crit = chi2.ppf(1 - alpha, df)
chi_crit

np.float64(12.591587243743977)

In [56]:
pval = chi2.sf(chi_stat, df)
print(f'{pval:.15f}')

0.000000000010462


**chi_stat (63.1141) > chi_crit (12.5916) -> Reject H0**
**p-value (< 0.0001) < 0.05 -> Reject H0**

> Note: chi-square test of independence is always right-tailed. The critical value uses chi2.ppf(1 - alpha, df), not 1 - alpha/2.

#### Step 7 (b) : Verify with scipy.stats.chi2_contingency()

chi2_contingency() computes the test statistic, p-value, degrees of freedom, and expected frequencies directly from the observed table. Should match our manual result.

In [57]:
chi_stat, pval, dof, Exp = chi2_contingency(O)
chi_stat, pval, dof, Exp

(np.float64(63.11414381079814),
 np.float64(1.04618387324474e-11),
 6,
 array([[50.41509434, 58.86792453, 50.71698113],
        [47.26415094, 55.18867925, 47.54716981],
        [40.96226415, 47.83018868, 41.20754717],
        [28.35849057, 33.11320755, 28.52830189]]))

#### Step 8 (c) : Standardized Residuals

$$SR_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

*Each SR is basically a z-score for that cell. Positive means observed > expected, negative means observed < expected. |SR| > 2 flags cells that are individually notable at ~5%.*

In [58]:
SR = (O - E) / np.sqrt(E)
SR.round(2)

array([[ 4.73, -1.03, -3.61],
       [-1.2 ,  1.59, -0.51],
       [-2.34,  0.02,  2.3 ],
       [-1.95, -0.71,  2.71]])

#### Step 9 (d) : Which Cells Drive the Significance?

In [59]:
department_labels = ['Engineering', 'Marketing', 'Finance', 'HR']
perf_labels       = ['Remote', 'Hybrid', 'Office']

print('Cells with |SR| > 2 :')
found = False
for i in range(r):
    for j in range(c):
        if abs(SR[i, j]) > 2:
            print(f'  {department_labels[i]} x {perf_labels[j]} : SR = {SR[i, j]:.4f}')
            found = True
if not found:
    print('  None')

Cells with |SR| > 2 :
  Engineering x Remote : SR = 4.7300
  Engineering x Office : SR = -3.6111
  Finance x Remote : SR = -2.3378
  Finance x Office : SR = 2.3044
  HR x Office : SR = 2.7095


Pattern: technical roles (Engineering) lean remote, operational/admin roles (Finance, HR) lean office.

#### Step 10 (e) : Cramer's V

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

*Dividing by min(r,c) - 1 keeps V in the [0, 1] range regardless of table size. For a 4 x 3 table, min(r,c) = 3, so the denominator is n * 2.*

In [60]:
k = min(r, c)

In [61]:
V = np.sqrt(chi_stat / (n * (k - 1)))
V

np.float64(0.24401156756294679)

#### Step 11 (f) : Effect Size Interpretation

Cohen's benchmarks for V depend on df* = min(r, c) - 1. For this table, df* = 2:

| Effect size | V threshold (df* = 2) |
|---|---|
| Small  | >= 0.07 |
| Medium | >= 0.21 |
| Large  | >= 0.35 |

In [62]:
# Cohen's thresholds for df* = min(r,c) - 1 = 2
small_t  = 0.10 / np.sqrt(k - 1)
medium_t = 0.30 / np.sqrt(k - 1)
large_t  = 0.50 / np.sqrt(k - 1)

print(f'Small  >= {small_t:.4f}')
print(f'Medium >= {medium_t:.4f}')
print(f'Large  >= {large_t:.4f}')
print(f"V        = {V:.4f}")

Small  >= 0.0707
Medium >= 0.2121
Large  >= 0.3536
V        = 0.2440


V = 0.2440, which falls between the medium (0.2121) and large (0.3536) thresholds, so the association is **moderate**.

Department explains a meaningful but not dominant portion of the variation in work mode preference.

#### Final Conclusion

| Metric | Value |
|---|---|
| Sample size $n$ | 530 |
| Table dimensions | 4 x 3 |
| All expected counts >= 5 | Yes |
| $\chi^2$ statistic | 63.1141 |
| Degrees of freedom | 6 |
| $\chi^2$ critical (right-tail, alpha = 0.05) | 12.5916 |
| p-value | < 0.0001 |
| Decision | **Reject H0** |
| Cells with |SR| > 2 | Engineering x Remote, Engineering x Office, Finance x Remote, Finance x Office, HR x Office |
| Cramer's V | 0.2440 |
| Effect size | **Moderate** |

There is significant evidence that work mode preference depends on department (chi2 = 63.11, p < 0.0001). Engineering strongly prefers Remote and avoids Office; Finance and HR lean toward Office. Cramer's V = 0.24 indicates a moderate association.

## 📌 Question 9 : Chi-Square Goodness-of-Fit for the Poisson Distribution

A cloud server company records the number of server crashes per day over 400 days.

| Crashes/Day | 0 | 1 | 2 | 3 | 4 | 5+ |
|---|---|---|---|---|---|---|
| Observed | 72 | 118 | 101 | 63 | 29 | 17 |

Assume crashes follow a Poisson distribution.

**(a)** Estimate $\lambda$ from the data.  
**(b)** Compute expected frequencies using the Poisson model.  
**(c)** Explain why categories may need to be combined.  
**(d)** Verify all chi-square assumptions.  
**(e)** Perform the chi-square goodness-of-fit test manually.  
**(f)** Repeat using `scipy.stats.chisquare()`.  
**(g)** Explain why df changes when $\lambda$ is estimated from the data.  
**(h)** Compute standardized residuals.  
**(i)** Identify which crash frequencies deviate most from the Poisson model.  
**(j)** Interpret the practical meaning of rejecting the Poisson model.

*The Poisson distribution models events occurring independently at a constant average rate. Here, we don't know the true crash rate, so we estimate $\lambda$ from the data itself. That estimation costs one extra degree of freedom beyond the usual $k - 1$, giving $df = k - 2$. The goodness-of-fit test then checks whether the observed frequency pattern is consistent with a fitted Poisson model.*

#### Step 1 : Hypotheses

$$H_0: \text{The number of daily crashes follows a Poisson distribution}$$
$$H_1: \text{The distribution is not Poisson}$$

#### Step 2 : Data Setup

In [63]:
X = np.array([0, 1, 2, 3, 4, 5])    # crash counts (5 represents 5+)
O = np.array([72, 118, 101, 63, 29, 17])

n = O.sum()
print('Total days n :', n)

Total days n : 400


#### Step 3 (a) : Estimate $\lambda$

$$\hat{\lambda} = \frac{\sum x_i \cdot O_i}{n}$$

*Since we don't know the true crash rate, we estimate $\lambda$ as the weighted mean of observed crash counts. This is the MLE for the Poisson parameter and what we plug into the Poisson PMF to get expected frequencies.*

In [64]:
# part a - weighted mean gives MLE of lambda
lam = np.sum(X * O) / n
print('Estimated lambda :', round(lam, 4))

Estimated lambda : 1.775


#### Step 4 (b) : Expected Frequencies

$$E_i = n \cdot P(X = i) = n \cdot \frac{e^{-\lambda} \lambda^i}{i!}$$

*For categories 0 through 4 we use the Poisson PMF. For "5+" we take the tail probability so all probabilities sum to 1. Expected counts are probabilities multiplied by n.*

In [65]:
# part b - Poisson probabilities for 0, 1, 2, 3, 4
probs = poisson.pmf([0, 1, 2, 3, 4], lam)

# 5+ gets the remaining tail probability (everything beyond 4)
prob_5plus = 1 - probs.sum()

E = np.append(probs, prob_5plus) * n

print('Poisson probs (0-4) :', probs.round(4))
print('Prob (5+)           :', round(prob_5plus, 4))
print()
print('Expected counts E   :', E.round(2))

Poisson probs (0-4) : [0.1695 0.3008 0.267  0.158  0.0701]
Prob (5+)           : 0.0346

Expected counts E   : [ 67.79 120.33 106.8   63.19  28.04  13.85]


#### Step 5 (c) : Why Categories May Need to Be Combined

The chi-square approximation requires all expected counts to be at least 5. If the Poisson tail probability is very small, the "5+" bin (or other extreme bins) can fall below that threshold. When that happens, adjacent categories are merged until all expected counts meet the condition.

We check this in the next step.

#### Step 6 (d) : Verify Chi-Square Assumptions

In [66]:
# part d - check if all expected counts >= 5
print('Expected counts :', E.round(2))
print()

if (E >= 5).all():
    print('All expected counts >= 5 : chi-square assumptions hold')
else:
    low = np.where(E < 5)[0]
    print(f'Assumption violated: categories {low} have expected count < 5')
    print('Consider merging adjacent categories before proceeding')

Expected counts : [ 67.79 120.33 106.8   63.19  28.04  13.85]

All expected counts >= 5 : chi-square assumptions hold


#### Step 7 (e) : Chi-Square Test (Manual)

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}, \quad df = k - 2$$

*Normally df = k - 1. But since we estimated $\lambda$ from the data (one parameter), we lose one more degree of freedom. General rule: $df = k - 1 - p$ where $p$ = number of parameters estimated.*

In [67]:
# part e
alpha = 0.05

# df = k - 1 (total constraint) - 1 (lambda estimated from data)
df = len(X) - 1 - 1
print('df :', df)

df : 4


In [68]:
chi_stat = ((O - E)**2 / E).sum()
print('chi_stat :', round(chi_stat, 4))

chi_stat : 1.3703


In [69]:
chi_crit = chi2.ppf(1 - alpha, df)
print('chi_crit :', round(chi_crit, 4))

chi_crit : 9.4877


In [70]:
pval = chi2.sf(chi_stat, df)
print('p-value  :', round(pval, 4))

p-value  : 0.8493


**chi_stat < chi_crit and p-value > 0.05 -- Fail to Reject $H_0$**

The observed crash frequencies are consistent with a Poisson model.

#### Step 8 (f) : Verify with `scipy.stats.chisquare()`

*`chisquare()` takes observed and expected counts directly. It uses $df = k - 1$ internally, so the p-value it returns will differ from our manual calculation (where $df = k - 2$ because $\lambda$ was estimated). The chi_stat values will match.*

In [71]:
# part f
chi_stat_scipy, pval_scipy = chisquare(O, E)
print('chi_stat (scipy) :', round(chi_stat_scipy, 4))
print('p-value  (scipy, df=k-1) :', round(pval_scipy, 4))
print()
print('Our manual p-value uses df = k-2 to account for estimating lambda.')

chi_stat (scipy) : 1.3703
p-value  (scipy, df=k-1) : 0.9275

Our manual p-value uses df = k-2 to account for estimating lambda.


#### Step 9 (g) : Why Estimating $\lambda$ Changes the Degrees of Freedom

In a standard goodness-of-fit test where the null distribution is fully specified (no unknown parameters), $df = k - 1$.

When we estimate $\lambda$ from the data, we use one piece of information from the observed frequencies to pin down the distribution. That costs one more degree of freedom:

$$df = k - 1 - p$$

where $p$ = number of parameters estimated. Here $p = 1$ (just $\lambda$), so $df = k - 2 = 4$.

If we used the chi-square test without reducing df, we would have an anticonservative test: it would reject $H_0$ less often than it should, producing inflated p-values.

#### Step 10 (h) : Standardized Residuals

$$SR_i = \frac{O_i - E_i}{\sqrt{E_i}}$$

*Each SR is a z-score for one category. Positive means the observed count is higher than the Poisson model predicts; negative means lower. |SR| > 2 flags a notable deviation.*

In [72]:
crash_labels = ['0 crashes', '1 crash', '2 crashes', '3 crashes', '4 crashes', '5+ crashes']

SR = (O - E) / np.sqrt(E)

print('Standardized Residuals:')
for label, sr in zip(crash_labels, SR):
    flag = ' <-- notable' if abs(sr) > 2 else ''
    print(f'  {label:13s} : SR = {sr:+.4f}{flag}')

Standardized Residuals:
  0 crashes     : SR = +0.5109
  1 crash       : SR = -0.2127
  2 crashes     : SR = -0.5608
  3 crashes     : SR = -0.0236
  4 crashes     : SR = +0.1814
  5+ crashes    : SR = +0.8462


#### Step 11 (i) : Which Categories Deviate Most?

In [73]:
# part i - flag categories where |SR| > 2
print('Categories with |SR| > 2 :')
found = False
for label, sr in zip(crash_labels, SR):
    if abs(sr) > 2:
        direction = 'more' if sr > 0 else 'fewer'
        print(f'  {label}: SR = {sr:.4f}  ({direction} crashes than Poisson predicts)')
        found = True
if not found:
    print('  None - no category deviates notably from the Poisson model')

Categories with |SR| > 2 :
  None - no category deviates notably from the Poisson model


#### Step 12 (j) : Practical Meaning of Rejecting the Poisson Model

If we had rejected $H_0$, it would mean the crash process does not behave like simple Poisson arrivals. Possible reasons include:

- **Clustering / burstiness**: crashes arrive in batches rather than independently, suggesting shared causes like overload events or cascading failures.
- **Non-constant rate**: $\lambda$ changes over time (deployments, traffic spikes), violating the Poisson assumption of a fixed rate.
- **Overdispersion**: if variance exceeds the mean, a negative binomial model would fit better.

From an operational standpoint, Poisson models are often used for capacity planning and SLA design. A poor fit means those plans rest on incorrect assumptions and may underestimate tail risk.

In this problem, we fail to reject $H_0$, so the Poisson model is a reasonable description of the crash process with $\hat{\lambda} \approx 1.83$ crashes per day.

#### Final Conclusion

| Metric | Value |
|---|---|
| Total days $n$ | 400 |
| Estimated $\hat{\lambda}$ | 1.7750 |
| Categories $k$ | 6 |
| Parameters estimated | 1 ($\lambda$) |
| Degrees of freedom | 4 (= k - 2) |
| $\chi^2$ statistic | 1.3703 |
| $\chi^2$ critical (right-tail, $\alpha = 0.05$, df = 4) | 9.4877 |
| p-value | 0.8493 |
| Decision | **Fail to Reject $H_0$** |
| Cells with $|SR| > 2$ | None |

The observed crash frequency distribution is consistent with a Poisson($\hat{\lambda} = 1.7750$) model. No individual crash-count category shows a notable deviation from what the Poisson model predicts. The company can reasonably use Poisson-based models for capacity planning and uptime estimation.

## 📌 Question 10 : Chi-Square Test of Homogeneity + Post-Hoc Analysis (AI Learning Systems)

Three universities implement different AI-assisted learning systems.
At the end of the semester, students are classified by performance level.

|  | Excellent | Good | Average | Poor | Row Total |
|---|---|---|---|---|---|
| University A | 61 | 74 | 39 | 16 | 190 |
| University B | 43 | 69 | 58 | 30 | 200 |
| University C | 27 | 52 | 81 | 40 | 200 |
| **Col Total** | **131** | **195** | **178** | **86** | **590** |

**(a)** Perform the chi-square test of homogeneity manually.

**(b)** Solve again using `scipy.stats.chi2_contingency()`.

**(c)** Compute standardized residuals for all cells.

**(d)** Identify the cells driving statistical significance.

**(e)** Apply Bonferroni correction for post-hoc residual testing.

**(f)** Compute Cramér's V.

**(g)** Explain whether statistical significance implies meaningful educational differences.

*The test of homogeneity asks whether the distribution of performance is the same across all three universities. The overall chi-square will tell us if there is a significant difference somewhere, but not where. Standardized residuals pin down the specific university-performance combinations that are unusual. With 12 cells being tested simultaneously, the Bonferroni correction is applied to avoid false positives. Cramer's V then gives the practical effect size.*

#### Step 1 : Hypotheses

$$H_0: \text{The distribution of performance is the same across all three universities}$$
$$H_1: \text{At least one university has a different performance distribution}$$

#### Step 2 : Data Setup and Expected Counts

$$E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{n}$$

*Under H0, each cell's expected count depends only on the marginals. `np.outer` computes all 12 at once.*

In [74]:
alpha = 0.05

O = np.array([[61, 74, 39, 16],   # University A
              [43, 69, 58, 30],   # University B
              [27, 52, 81, 40]])  # University C

row_total = O.sum(axis=1)
col_total = O.sum(axis=0)
n = O.sum()

print('Row totals :', row_total)
print('Col totals :', col_total)
print('n          :', n)

Row totals : [190 200 200]
Col totals : [131 195 178  86]
n          : 590


In [75]:
E = np.outer(row_total, col_total) / n

print('Expected counts:')
print(E.round(2))
print()
print('All expected counts >= 5 :', (E >= 5).all())

Expected counts:
[[42.19 62.8  57.32 27.69]
 [44.41 66.1  60.34 29.15]
 [44.41 66.1  60.34 29.15]]

All expected counts >= 5 : True


#### Step 3 : Degrees of Freedom

$$df = (r - 1)(c - 1) = (3 - 1)(4 - 1) = 6$$

In [76]:
r, c = O.shape
df = (r-1) * (c-1)
print('df :', df)

df : 6


#### Step 4 (a) : Chi-Square Test (Manual)

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [77]:
chi_stat = ((O - E)**2 / E).sum()
chi_crit = chi2.ppf(1 - alpha, df)
pval = chi2.sf(chi_stat, df)

print('chi_stat :', round(chi_stat, 4))
print('chi_crit :', round(chi_crit, 4))
print('p-value  :', round(pval, 6))

chi_stat : 42.4132
chi_crit : 12.5916
p-value  : 0.0


**chi_stat (42.4132) > chi_crit (12.5916) -- Reject H0**

Performance distribution is not the same across all three universities.

#### Step 5 (b) : Verify with `scipy.stats.chi2_contingency()`

`chi2_contingency()` takes the observed table and returns the statistic, p-value, df, and expected counts directly. Should match our manual result.

In [78]:
chi_stat, pval, df, exp = chi2_contingency(O)
chi_stat, pval, df, exp

(np.float64(42.41315204983978),
 np.float64(1.523751172653974e-07),
 6,
 array([[42.18644068, 62.79661017, 57.3220339 , 27.69491525],
        [44.40677966, 66.10169492, 60.33898305, 29.15254237],
        [44.40677966, 66.10169492, 60.33898305, 29.15254237]]))

#### Step 6 (c) : Standardized Residuals

$$SR_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij}}}$$

*Each SR is a z-score for that cell. Positive means observed > expected, negative means observed < expected. The chi-square test told us there is a significant difference somewhere -- residuals tell us exactly where.*

In [79]:
SR = (O - E) / np.sqrt(E)
print(SR.round(4))

[[ 2.8966  1.4138 -2.42   -2.2223]
 [-0.2111  0.3565 -0.3011  0.157 ]
 [-2.6121 -1.7345  2.6598  2.009 ]]


#### Step 7 (d) : Which Cells Drive the Significance?

In [80]:
Uni = ['University A', 'University B', 'University C']
perf_labels = ['Excellent', 'Good', 'Average', 'Poor']

print('Cells with |SR| > 2 :')
found = False
for i in range(r):
    for j in range(c):
        if abs(SR[i, j]) > 2:
            print(f'  {Uni[i]} x {perf_labels[j]} : SR = {SR[i, j]:.4f}')
            found = True
if not found:
    print('  None')

Cells with |SR| > 2 :
  University A x Excellent : SR = 2.8966
  University A x Average : SR = -2.4200
  University A x Poor : SR = -2.2223
  University C x Excellent : SR = -2.6121
  University C x Average : SR = 2.6598
  University C x Poor : SR = 2.0090


#### Step 8 (e) : Bonferroni Correction for Post-Hoc Testing

When we check all 12 cells individually for significance, the chance of at least one false positive grows. Bonferroni correction divides alpha by the number of cells, giving a stricter per-cell threshold.

$$\alpha_{\text{Bonf}} = \frac{\alpha}{\text{total cells}} = \frac{0.05}{12}$$

The corrected critical value is the z-score corresponding to $\alpha_{\text{Bonf}}/2$ in each tail.

In [81]:
# part e - bonferroni correction
total_cells = r * c
bonf_alpha = alpha / total_cells
bonf_crit = norm.ppf(1 - bonf_alpha / 2)

print('Total cells     :', total_cells)
print('Bonf alpha      :', round(bonf_alpha, 5))
print('Bonf crit value :', round(bonf_crit, 4))  # cells need |SR| > this to be significant after correction

Total cells     : 12
Bonf alpha      : 0.00417
Bonf crit value : 2.8653


In [82]:
# cells that survive bonferroni correction (stricter threshold)
sig_cells = abs(SR) > bonf_crit

sig_df = pd.DataFrame(
    sig_cells,
    index=Uni,
    columns=perf_labels
)
sig_df

,Excellent,Good,Average,Poor
University A,True,False,False,False
University B,False,False,False,False
University C,False,False,False,False


#### Step 9 (f) : Cramér's V

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

*For a 3 x 4 table, min(r, c) = 3 and df* = 2. Cohen's thresholds at df* = 2 are: small >= 0.07, medium >= 0.21, large >= 0.35.*

In [83]:
k = min(r, c)
V = np.sqrt(chi_stat / (n * (k - 1)))
print('Cramer\'s V :', round(V, 4))

Cramer's V : 0.1896


In [84]:
# Cohen's thresholds for df* = min(r,c) - 1 = 2
small_t  = 0.10 / np.sqrt(k - 1)
medium_t = 0.30 / np.sqrt(k - 1)
large_t  = 0.50 / np.sqrt(k - 1)

print(f'Small  >= {small_t:.4f}')
print(f'Medium >= {medium_t:.4f}')
print(f'Large  >= {large_t:.4f}')
print(f'V        = {V:.4f}')

Small  >= 0.0707
Medium >= 0.2121
Large  >= 0.3536
V        = 0.1896


#### Step 10 (g) : Statistical Significance vs Practical/Educational Meaning

Statistical significance tells us that the performance distributions across universities are unlikely to be identical by chance (p < 0.05). But it does not tell us:

- **How large the difference is**: Cramer's V of 0.1896 suggests a small-to-moderate association, not a large one.
- **Why the differences exist**: The AI system is one possible explanation, but class sizes, student intake quality, grading standards, or instructor quality could also drive the pattern.
- **Whether it matters in practice**: A university with 5% more students in "Average" vs "Excellent" may not justify changing its entire system.

The residual analysis showed University A produces more Excellent students than expected, and University C produces more Average/Poor students. This is statistically real, but whether it reflects the AI system or something else entirely requires further investigation beyond the chi-square test.

#### Final Conclusion

| Metric | Value |
|---|---|
| Sample size $n$ | 590 |
| Table dimensions | 3 x 4 |
| All expected counts >= 5 | Yes |
| $\chi^2$ statistic | 42.4132 |
| Degrees of freedom | 6 |
| $\chi^2$ critical (alpha = 0.05) | 12.5916 |
| p-value | < 0.0001 |
| Decision | **Reject H0** |
| Cells with |SR| > 2 | Uni A x Excellent, Uni A x Average, Uni A x Poor, Uni C x Excellent, Uni C x Average, Uni C x Poor |
| Bonferroni threshold (|SR|) | 2.8653 |
| Cells surviving Bonferroni | University A x Excellent only |
| Cramér's V | 0.1896 |
| Effect size | **Small to moderate** |

There is significant evidence that performance distributions differ across universities (chi2 = 42.41, p < 0.0001). University A produces more Excellent students than expected; University C shows the opposite pattern (more Average and Poor). After Bonferroni correction (threshold |SR| > 2.87), only University A x Excellent remains significant. Cramer's V = 0.19 indicates a small-to-moderate practical effect.